Sequential vs. concurrent. Call fake_call 10 times in a plain for loop with await, and time it. Then time the gather version. Write your predictions down before running.

In [7]:
import asyncio, time

predictions
sequential version: 10 secs
gather version: 1 sec

In [20]:
start = time.time()

results = []

for i in range(10):
    r = await fake_call(i)
    results.append(r)

print(results)
print(f"{time.time() - start:.1f}s")

['result 0', 'result 1', 'result 2', 'result 3', 'result 4', 'result 5', 'result 6', 'result 7', 'result 8', 'result 9']
10.0s


Real APIs block you if you fire 200 requests at once, so you need a limit.

In [24]:
sem = asyncio.Semaphore(5)

async def limited_call(i):
    async with sem:
        r = await fake_call(i)
        return r

start = time.time()
results = await asyncio.gather(*(limited_call(i) for i in range(12)))
print (results[:5])
print(f"{time.time() - start:.1f}s")

['result 0', 'result 1', 'result 2', 'result 3', 'result 4']
3.0s


Exercise 3

In [45]:
import random
import json

async def fake_call(i):
    await asyncio.sleep(1)
    if random.random() < 0.2:
        raise Exception(f"call {i} failed")
    return f"result {i}"

sem = asyncio.Semaphore(5)

async def limited_call(i):
    async with sem:
        for attempt in range(3):
            try:
                r = {"id": i, "response": await fake_call(i), "error": None, "attempts": attempt+1}
                break
            except Exception as e:
                r = {"id": i, "response": None, "error": f"error: {e}", "attempts": attempt+1}
        return r

start = time.time()
results = await asyncio.gather(*(limited_call(i) for i in range(20)))
print (results)
print(f"{time.time() - start:.1f}s")

[{'id': 0, 'response': 'result 0', 'error': None, 'attempts': 1}, {'id': 1, 'response': 'result 1', 'error': None, 'attempts': 1}, {'id': 2, 'response': 'result 2', 'error': None, 'attempts': 1}, {'id': 3, 'response': 'result 3', 'error': None, 'attempts': 1}, {'id': 4, 'response': 'result 4', 'error': None, 'attempts': 1}, {'id': 5, 'response': 'result 5', 'error': None, 'attempts': 1}, {'id': 6, 'response': 'result 6', 'error': None, 'attempts': 1}, {'id': 7, 'response': 'result 7', 'error': None, 'attempts': 1}, {'id': 8, 'response': 'result 8', 'error': None, 'attempts': 1}, {'id': 9, 'response': 'result 9', 'error': None, 'attempts': 2}, {'id': 10, 'response': 'result 10', 'error': None, 'attempts': 1}, {'id': 11, 'response': 'result 11', 'error': None, 'attempts': 1}, {'id': 12, 'response': 'result 12', 'error': None, 'attempts': 1}, {'id': 13, 'response': 'result 13', 'error': None, 'attempts': 3}, {'id': 14, 'response': 'result 14', 'error': None, 'attempts': 2}, {'id': 15, 're

In [56]:
with open("results.jsonl", "a") as f:
    for record in results:
        f.write(json.dumps(record) + ("\n"))

In [59]:
import pandas as pd
df = pd.read_json("results.jsonl", lines = True)
print(df)
print(len(df))

    id   response  error  attempts
0    0   result 0    NaN         1
1    1   result 1    NaN         1
2    2   result 2    NaN         1
3    3   result 3    NaN         1
4    4   result 4    NaN         1
5    5   result 5    NaN         1
6    6   result 6    NaN         1
7    7   result 7    NaN         1
8    8   result 8    NaN         1
9    9   result 9    NaN         2
10  10  result 10    NaN         1
11  11  result 11    NaN         1
12  12  result 12    NaN         1
13  13  result 13    NaN         3
14  14  result 14    NaN         2
15  15  result 15    NaN         1
16  16  result 16    NaN         1
17  17  result 17    NaN         2
18  18  result 18    NaN         1
19  19  result 19    NaN         1
20


In [60]:
(df["attempts"] > 1).sum()

np.int64(4)

In [83]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
load_dotenv()
client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1",
                api_key=os.environ["OPENROUTER_API_KEY"])

MODEL = "qwen/qwen3.8-flash"
PROMPT = "Name one random animal. Reply with just the word."

async def real_call(prompt):
    resp = await client.chat.completions.create(
    model= MODEL,
    messages=[{"role": "user", "content": prompt}],
    temperature=1,
    )
    return resp.choices[0].message.content

print(await real_call("Say the word banana and nothing else."))



banana


In [84]:
sem = asyncio.Semaphore(5)

async def limited_call(i):
    async with sem:
        for attempt in range(3):
            try:
                r = {"id": i, "response": await real_call(PROMPT), "error": None, "attempts": attempt+1, "model": MODEL, "prompt": PROMPT}
                break
            except Exception as e:
                r = {"id": i, "response": None, "error": f"error: {e}", "attempts": attempt+1, "model": MODEL, "prompt": PROMPT}
        return r

start = time.time()
results = await asyncio.gather(*(limited_call(i) for i in range(20)))
print (results)
print(f"{time.time() - start:.1f}s")

[{'id': 0, 'response': 'Okapi', 'error': None, 'attempts': 1, 'model': 'qwen/qwen3.8-flash', 'prompt': 'Name one random animal. Reply with just the word.'}, {'id': 1, 'response': 'otter', 'error': None, 'attempts': 1, 'model': 'qwen/qwen3.8-flash', 'prompt': 'Name one random animal. Reply with just the word.'}, {'id': 2, 'response': 'octopus', 'error': None, 'attempts': 1, 'model': 'qwen/qwen3.8-flash', 'prompt': 'Name one random animal. Reply with just the word.'}, {'id': 3, 'response': 'platypus', 'error': None, 'attempts': 1, 'model': 'qwen/qwen3.8-flash', 'prompt': 'Name one random animal. Reply with just the word.'}, {'id': 4, 'response': 'Otter', 'error': None, 'attempts': 1, 'model': 'qwen/qwen3.8-flash', 'prompt': 'Name one random animal. Reply with just the word.'}, {'id': 5, 'response': 'Otter', 'error': None, 'attempts': 1, 'model': 'qwen/qwen3.8-flash', 'prompt': 'Name one random animal. Reply with just the word.'}, {'id': 6, 'response': 'otter', 'error': None, 'attempts': 

In [86]:
with open("real_results.jsonl", "a") as f:
    for record in results:
        f.write(json.dumps(record) + ("\n"))

In [89]:
df = pd.read_json("real_results.jsonl", lines = True)
df["response"].str.lower().str.strip().value_counts()

response
otter       7
axolotl     5
platypus    3
okapi       2
octopus     2
fox         1
Name: count, dtype: int64